# Phase 1 — Glacier-algal pigment photophysics (Level 1 & Level 2)

B3LYP/6-31G(d) + IEF-PCM(water, ε = 78.4) geometry → TD-DFT (30 singlets, B3LYP and CAM-B3LYP) → Gaussian-broadened (FWHM 0.3 eV) **MAC(λ), 300–800 nm**.

**Runtime:** a CPU runtime works but is slow. On 2 vCPUs, Level 1 takes a few hours and Level 2 is a multi-session job. The optimiser checkpoints every step (`*_last.xyz`), so resume with `--start-xyz`. A GPU runtime enables `--gpu` (gpu4pyscf), which is much faster; see README → Runtime.

In [ ]:
# 1) Install (~1-2 min). gpu4pyscf is only used if a GPU runtime is attached.
import subprocess, sys
gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
pkgs = ["pyscf>=2.8", "geometric", "rdkit", "tblite", "pandas", "scipy", "matplotlib"]
if gpu:
    pkgs.append("gpu4pyscf-cuda12x")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)
print("GPU runtime:", gpu)

In [ ]:
# 2) Get the code (replace with your repo URL / branch, or upload the phase1/ folder)
!git clone -q -b claude/sweet-noether-6pryo4 https://github.com/avinautiya/algae.git || true
%cd algae/phase1

In [ ]:
# 3) (Optional) persist results to Google Drive so a disconnect does not lose the optimisation
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/algae_phase1'

In [ ]:
# 4) Level 1 - Purpurogallin (C11H8O5, charge 0, singlet)
GPU_FLAG = "--gpu" if gpu else ""
!python run_phase1.py --level level1 {GPU_FLAG} --tddft-functionals B3LYP CAM-B3LYP --outdir {OUT}/level1

In [ ]:
# 5) Level 2 - Purpurogallin carboxylic acid-6-O-beta-D-glucopyranoside (C18H18O12, charge 0, singlet)
#    Resume after a disconnect with:  --start-xyz {OUT}/level2/opt_B3LYP_last.xyz
!python run_phase1.py --level level2 {GPU_FLAG} --tddft-functionals B3LYP CAM-B3LYP --outdir {OUT}/level2

In [ ]:
# 6) Load the plot-ready outputs
import pandas as pd, numpy as np, matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 4))
for level, ls in [("level1", "-"), ("level2", "--")]:
    for func, c in [("B3LYP", "#7a2e0e"), ("CAM-B3LYP", "#1f5f8b")]:
        try:
            df = pd.read_csv(f"{OUT}/{level}/{level}_{func}_spectrum.csv")
        except FileNotFoundError:
            continue
        ax.plot(df.Wavelength_nm, df.MAC_estimated, ls, color=c, label=f"{level} TD-{func}")
ax.set_xlabel("Wavelength (nm)"); ax.set_ylabel("MAC (m$^2$ kg$^{-1}$)"); ax.set_xlim(300, 800); ax.legend()
plt.show()
pd.read_csv(f"{OUT}/level1/level1_B3LYP_states.csv").head(15)